# `boxsdk` smoke test — `box_sdk_gen` (Box next-gen SDK)

PyPI `boxsdk` v10+ **is** Box's generated next-gen SDK: upstream retired the legacy
`boxsdk` 3.x API (`from boxsdk import Client, OAuth2`) and re-used the PyPI name for the
generated SDK, changing the **import name to `box_sdk_gen`**.

All cells run **offline** (no Box account, no network): client construction + manager
surface, developer-token auth, token storage, schema serialization round-trip, the error
hierarchy, and immutable client re-configuration.

In [ ]:
# Example 1: Construct the client (offline). PyPI `boxsdk` v10+ is Box's generated
# next-gen SDK: the legacy `boxsdk` (Client/OAuth2) API is gone; auth classes are
# BoxDeveloperTokenAuth / BoxOAuth / BoxCCGAuth / BoxJWTAuth, one manager per resource group.
from box_sdk_gen import BoxClient, BoxDeveloperTokenAuth

auth = BoxDeveloperTokenAuth(token="dev-token-not-real")
client = BoxClient(auth=auth)

expected_managers = ["files", "folders", "uploads", "downloads", "users", "search",
                     "user_collaborations", "events", "webhooks", "chunked_uploads"]
missing = [m for m in expected_managers if not hasattr(client, m)]
assert not missing, f"missing manager attributes: {missing}"
public_attrs = [m for m in dir(client) if not m.startswith('_')]
print(f"BoxClient constructed offline; {len(public_attrs)} public attributes")
print("managers verified:", ", ".join(expected_managers))

In [ ]:
# Example 2: Developer-token auth + token storage — a full local round-trip (the
# developer token is wrapped into an AccessToken without any network call).
from box_sdk_gen import AccessToken, BoxDeveloperTokenAuth, InMemoryTokenStorage

auth = BoxDeveloperTokenAuth(token="dev-token-123")
tok = auth.retrieve_token()
assert isinstance(tok, AccessToken) and tok.access_token == "dev-token-123"

storage = InMemoryTokenStorage()
storage.store(tok)
assert storage.get().access_token == "dev-token-123"
storage.clear()
assert storage.get() is None
print("developer-token auth + InMemoryTokenStorage round-trip OK")

In [ ]:
# Example 3: Schema serialization round-trip — generated schema objects serialize to
# plain JSON-safe dicts and deserialize back (the SDK's core wire-format invariant).
from box_sdk_gen import User, serialize, deserialize

user = User(id="123", name="Test User", login="test@example.com")
data = serialize(user)
print("serialized:", data)
assert data == {"id": "123", "type": "user", "name": "Test User", "login": "test@example.com"}

restored = deserialize(data, User)
assert isinstance(restored, User)
assert (restored.id, restored.name, restored.login) == ("123", "Test User", "test@example.com")
print("deserialize(serialize(user)) round-trip OK")

In [ ]:
# Example 4: Error hierarchy — BoxSDKError is the SDK base exception; BoxAPIError
# (HTTP-layer failures) subclasses it, so one except-clause covers both.
from box_sdk_gen import BoxAPIError, BoxSDKError

assert issubclass(BoxSDKError, Exception)
assert issubclass(BoxAPIError, BoxSDKError)

try:
    raise BoxSDKError(message="simulated SDK failure")
except BoxSDKError as e:
    caught = e
assert caught.message == "simulated SDK failure"
print(f"str(error) = {str(caught)!r}")
print(f"error hierarchy OK: BoxAPIError -> BoxSDKError -> Exception; message={caught.message!r}")

In [ ]:
# Example 5: Immutable client derivation — with_extra_headers / with_custom_base_urls
# return reconfigured copies (per-request config layering) without mutating the original.
from box_sdk_gen import BaseUrls, BoxClient, BoxDeveloperTokenAuth

client = BoxClient(auth=BoxDeveloperTokenAuth(token="dev-token-not-real"))

traced = client.with_extra_headers(extra_headers={"X-Trace-Id": "run-42"})
assert traced is not client
assert traced.network_session.additional_headers.get("X-Trace-Id") == "run-42"

rebased = client.with_custom_base_urls(base_urls=BaseUrls(base_url="https://box.example.test"))
assert rebased.network_session.base_urls.base_url == "https://box.example.test"
assert client.network_session.base_urls.base_url == "https://api.box.com"
print("client derivation OK: headers + base URLs layered without mutating the original")